# Blood Vessel Extraction & Segmentation

This notebook provides interactive inspection, step-by-step visualization, and quantitative evaluation of morphological blood vessel extraction algorithms on the **Fundus-AVSeg** dataset (*Nature Scientific Data, 2025*).

### Algorithms Evaluated:
1. **Method 1**: Local Adaptive Thresholding + Morphological Opening + Small Object Removal
2. **Method 2**: Multi-Scale Frangi Vesselness Filter + Mean Thresholding + Opening/Closing Morphological Operations

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skimage import io

# Import modular processing functions
from vessel_extraction import (
    load_split,
    load_image,
    load_annotation,
    annotation_to_binary_mask,
    get_disease_label,
    extract_green_channel,
    segment_adaptive_threshold,
    segment_frangi,
    evaluate_segmentation,
    compute_vessel_metrics,
    compute_branch_analysis,
    process_single_image,
    process_dataset,
    RESULTS_DIR
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## 1. Dataset Overview & Split Information
Inspect the dataset splits defined in `training.txt` (80 images) and `testing.txt` (20 images).

In [ ]:
train_files = load_split('train')
test_files = load_split('test')

print(f"Training set size: {len(train_files)} images")
print(f"Testing set size:  {len(test_files)} images")

# Breakdown by disease category in test set
disease_counts = {}
for f in test_files:
    label = get_disease_label(f)
    disease_counts[label] = disease_counts.get(label, 0) + 1

print("\nTest set breakdown by disease:")
for disease, count in disease_counts.items():
    print(f" - {disease}: {count} images")

## 2. Sample Image & Ground Truth Annotation Inspection
Fundus-AVSeg ground truth annotations are color-coded (Red=Arteries, Blue=Veins, Green=Crossings, White=Uncertain).

In [ ]:
sample_file = test_files[0] if test_files else '035_A.png'
img = load_image(sample_file)
ann = load_annotation(sample_file)
gt_mask = annotation_to_binary_mask(ann)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(img)
axes[0].set_title(f"Fundus Image ({sample_file})")
axes[0].axis('off')

if ann is not None:
    axes[1].imshow(ann)
    axes[1].set_title("Multi-Color Annotation")
    axes[1].axis('off')

if gt_mask is not None:
    axes[2].imshow(gt_mask, cmap='gray')
    axes[2].set_title("Converted Binary Vessel Mask")
    axes[2].axis('off')

plt.tight_layout()
plt.show()

## 3. Step-by-Step Segmentation Walkthrough
Demonstrate green channel extraction, Method 1 (Adaptive Thresholding), and Method 2 (Frangi Filter).

In [ ]:
green = extract_green_channel(img)
m1_result = segment_adaptive_threshold(green)
m2_result = segment_frangi(green)

eval_m1 = evaluate_segmentation(m1_result, gt_mask)
eval_m2 = evaluate_segmentation(m2_result, gt_mask)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes[0, 0].imshow(green, cmap='gray')
axes[0, 0].set_title("1. Green Channel (Contrast Enhanced Area)")
axes[0, 0].axis('off')

axes[0, 1].imshow(gt_mask, cmap='gray')
axes[0, 1].set_title("2. Binary Ground Truth")
axes[0, 1].axis('off')

axes[1, 0].imshow(m1_result, cmap='gray')
axes[1, 0].set_title(f"Method 1: Local Adaptive (Dice: {eval_m1.get('dice', 0):.4f})")
axes[1, 0].axis('off')

axes[1, 1].imshow(m2_result, cmap='gray')
axes[1, 1].set_title(f"Method 2: Frangi Filter (Dice: {eval_m2.get('dice', 0):.4f})")
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

## 4. Vessel Structural Analysis & Network Graph
Extract vessel skeleton, measure vessel lengths, and map branch networks using `sknw`.

In [ ]:
v_metrics = compute_vessel_metrics(m1_result)
thin_mask = v_metrics['thin_mask']
b_analysis = compute_branch_analysis(thin_mask)

print(f"Total Thinned Vessel Length: {v_metrics['thinned_length']:.2f} pixels")
print(f"Length of Wide Vessels (>15px): {v_metrics['length_width_gt_15']:.2f} pixels")
print(f"Length of Major Vessels (>40px): {v_metrics['length_width_gt_40']:.2f} pixels")
print(f"Total Detected Branches: {b_analysis['num_branches']}")

# Graph Visualization
graph = b_analysis['graph']
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(green, cmap='gray')

if graph is not None:
    for (s, e) in graph.edges():
        pts = graph[s][e]['pts']
        ax.plot(pts[:, 1], pts[:, 0], 'lime', linewidth=1.5)
        
    nodes = graph.nodes()
    ps = np.array([nodes[i]['o'] for i in nodes])
    if len(ps) > 0:
        ax.plot(ps[:, 1], ps[:, 0], 'red', marker='o', linestyle='', markersize=3)

ax.set_title("Vessel Skeleton Network Graph (Green=Branches, Red=Nodes)")
ax.axis('off')
plt.show()

## 5. Branch Direction Analysis (Polar Rose Plot)
Plot distribution of branch orientations extracted from region properties.

In [ ]:
orientations = b_analysis['branch_orientations']
if len(orientations) > 0:
    n = len(orientations)
    angles = np.linspace(0, 2*np.pi, n+1)[:-1]
    
    fig = plt.figure(figsize=(6, 6))
    ax = fig.add_subplot(111, projection='polar')
    bars = ax.bar(angles, orientations, width=0.4)
    ax.set_title('Branch Orientation Polar Rose Plot', pad=15)
    labels = ['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW']
    ax.set_xticklabels(labels)
    ax.set_theta_offset(np.pi/2)
    
    for i, bar in enumerate(bars):
        bar.set_facecolor(plt.cm.viridis(i/n))
        
    plt.show()
else:
    print("No branches detected for polar plot.")

## 6. Full Test Set Evaluation & Disease Category Breakdown
Run batch evaluation on test split (or load pre-computed metrics CSV from `results/metrics_test.csv`).

In [ ]:
csv_path = os.path.join(RESULTS_DIR, 'metrics_test.csv')
if os.path.exists(csv_path):
    df_results = pd.read_csv(csv_path)
else:
    print("Running batch evaluation on test split...")
    df_results = process_dataset(split='test', save_vis=False)

# Display Results Table
display(df_results.head(10))

print("\n--- Aggregate Performance across Test Set ---")
print(f"Method 1 (Adaptive) Mean Dice: {df_results['m1_dice'].mean():.4f} +/- {df_results['m1_dice'].std():.4f}")
print(f"Method 2 (Frangi)   Mean Dice: {df_results['m2_dice'].mean():.4f} +/- {df_results['m2_dice'].std():.4f}")

# Breakdown by Disease
disease_summary = df_results.groupby('disease')[['m1_dice', 'm2_dice', 'm1_precision', 'm1_recall']].mean()
print("\n--- Performance Breakdown by Disease Category ---")
display(disease_summary)